 **6. Use the following small text dataset to train a simple Variational
Autoencoder (VAE) for text reconstruction:**

["The sky is blue", "The sun is bright", "The grass is green",
"The night is dark", "The stars are shining"]

1. Preprocess the data (tokenize and pad the sequences).
2. Build a basic VAE model for text reconstruction.
3. Train the model and show how it reconstructs or generates similar sentences.

In [3]:
!pip install tensorflow numpy

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# 1. PREPROCESSING THE DATA
texts = ["The sky is blue", "The sun is bright", "The grass is green",
         "The night is dark", "The stars are shining"]

# Lowercase and tokenize by splitting on spaces
tokens = [t.lower().split() for t in texts]

# Build Vocabulary with special tokens
vocab = {"<pad>": 0, "<sos>": 1, "<eos>": 2}
for seq in tokens:
    for word in seq:
        if word not in vocab:
            vocab[word] = len(vocab)

idx2word = {i: w for w, i in vocab.items()}
vocab_size = len(vocab)

# Pad sequences and add Start-Of-Sequence (<sos>) and End-Of-Sequence (<eos>)
max_len = max(len(seq) for seq in tokens)
data = []
for seq in tokens:
    encoded = [vocab["<sos>"]] + [vocab[w] for w in seq] + [vocab["<eos>"]]
    # Add padding to make all sentences the same length
    encoded += [vocab["<pad>"]] * (max_len - len(seq))
    data.append(encoded)

# Convert to PyTorch tensor
data_tensor = torch.tensor(data) # Shape: (5, max_len + 2)


# 2. BUILDING THE VAE MODEL
class TextVAE(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=32, latent_dim=8):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # Encoder: reads the sequence and outputs a hidden state
        self.encoder_gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)

        # Latent Space: maps hidden state to Mean (mu) and Variance (log_var)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_var = nn.Linear(hidden_dim, latent_dim)

        # Decoder: maps latent space back to hidden state to generate text
        self.fc_z = nn.Linear(latent_dim, hidden_dim)
        self.decoder_gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def encode(self, x):
        embedded = self.embed(x)
        _, hidden = self.encoder_gru(embedded)
        hidden = hidden.squeeze(0) # Remove RNN layer dimension
        mu = self.fc_mu(hidden)
        log_var = self.fc_var(hidden)
        return mu, log_var

    def reparameterize(self, mu, log_var):
        # Trick to allow backpropagation through random sampling
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        # 1. Encode
        mu, log_var = self.encode(x)
        # 2. Sample from latent space
        z = self.reparameterize(mu, log_var)

        # 3. Decode (using Teacher Forcing for training)
        hidden = self.fc_z(z).unsqueeze(0)
        dec_input = x[:, :-1] # Input everything except the last token
        embedded = self.embed(dec_input)
        out, _ = self.decoder_gru(embedded, hidden)
        logits = self.fc_out(out)

        return logits, mu, log_var

    def generate(self, z, max_len):
        # Generate text iteratively from a latent vector (z)
        hidden = self.fc_z(z).unsqueeze(0)
        curr_token = torch.tensor([[vocab["<sos>"]]])
        generated_words = []

        for _ in range(max_len):
            embedded = self.embed(curr_token)
            out, hidden = self.decoder_gru(embedded, hidden)
            logits = self.fc_out(out)

            # Get the most likely next word
            next_token = logits.argmax(-1)
            curr_token = next_token

            word_idx = next_token.item()
            if word_idx == vocab["<eos>"]: break
            generated_words.append(idx2word[word_idx])

        return " ".join(generated_words)


# 3. TRAINING AND RECONSTRUCTION
model = TextVAE(vocab_size)
optimizer = optim.Adam(model.parameters(), lr=0.01)

print("Starting Training...")
for epoch in range(1, 201):
    optimizer.zero_grad()
    logits, mu, log_var = model(data_tensor)

    # Target is shifted by 1 (we predict the next word)
    targets = data_tensor[:, 1:]

    # Reconstruction Loss (Cross Entropy)
    recon_loss = F.cross_entropy(logits.reshape(-1, vocab_size), targets.reshape(-1), ignore_index=0)

    # KL Divergence Loss (forces latent space to be a normal distribution)
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / data_tensor.size(0)

    # Total Loss (using a small weight for KL to prevent posterior collapse)
    loss = recon_loss + 0.1 * kl_loss

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}/200 | Loss: {loss.item():.4f}")

print("\n--- Reconstructing Original Sentences ---")
with torch.no_grad():
    for i in range(len(data_tensor)):
        x = data_tensor[i:i+1]
        mu, _ = model.encode(x)
        # Using mu directly for deterministic reconstruction
        reconstructed = model.generate(mu, max_len=max_len+2)
        print(f"Original:      {texts[i]}")
        print(f"Reconstructed: {reconstructed}\n")

Starting Training...
Epoch 50/200 | Loss: 0.3295
Epoch 100/200 | Loss: 0.2789
Epoch 150/200 | Loss: 0.2608
Epoch 200/200 | Loss: 0.2734

--- Reconstructing Original Sentences ---
Original:      The sky is blue
Reconstructed: the sky is blue

Original:      The sun is bright
Reconstructed: the sun is bright

Original:      The grass is green
Reconstructed: the grass is green

Original:      The night is dark
Reconstructed: the night is dark

Original:      The stars are shining
Reconstructed: the stars are shining



**7. Use a pre-trained GPT model (like GPT-2 or GPT-3) to translate a short
English paragraph into French and German. Provide the original and translated text.**


In [9]:
!pip install transformers torch

In [19]:
from transformers import pipeline, set_seed

# Initialize the text-generation pipeline with the local GPT-2 model
# This will download the open-source model to your machine the first time you run it
generator = pipeline('text-generation', model='gpt2')
set_seed(42)

original_text = (
    "Artificial intelligence is transforming the way we interact with technology. "
    "From natural language processing to computer vision, these advancements are "
    "making our daily tasks easier and more efficient."
)

# We use "Few-Shot Prompting" to teach GPT-2 how to translate on the fly
french_prompt = (
    "Translate English to French.\n"
    "English: Hello, how are you?\n"
    "French: Bonjour, comment allez-vous?\n"
    "English: " + original_text + "\n"
    "French:"
)

german_prompt = (
    "Translate English to German.\n"
    "English: Hello, how are you?\n"
    "German: Hallo, wie geht es dir?\n"
    "English: " + original_text + "\n"
    "German:"
)

# Generate the translations
print("Generating French translation...")
french_output = generator(french_prompt, max_new_tokens=50, num_return_sequences=1, pad_token_id=50256)

print("Generating German translation...")
german_output = generator(german_prompt, max_new_tokens=50, num_return_sequences=1, pad_token_id=50256)

# Extract only the newly generated text
french_text = french_output[0]['generated_text'].split("French:")[-1].strip()
german_text = german_output[0]['generated_text'].split("German:")[-1].strip()

print("\n--- Original English Text ---")
print(original_text)
print("\n--- French Translation ---")
print(french_text)
print("\n--- German Translation ---")
print(german_text)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating French translation...


[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating German translation...

--- Original English Text ---
Artificial intelligence is transforming the way we interact with technology. From natural language processing to computer vision, these advancements are making our daily tasks easier and more efficient.

--- French Translation ---
I'm

--- German Translation ---
I would like to know your name.


In [24]:
from transformers import pipeline

# We use the standard 'text-generation' pipeline with a modern, small Causal LM
# Qwen2.5-0.5B-Instruct is a tiny open-source model that excels at instructions
translator = pipeline('text-generation', model='Qwen/Qwen2.5-0.5B-Instruct')

original_text = (
    "Artificial intelligence is transforming the way we interact with technology. "
    "From natural language processing to computer vision, these advancements are "
    "making our daily tasks easier and more efficient."
)

messages_fr = [
    {"role": "system", "content": "You are a professional translator."},
    {"role": "user", "content": f"Translate this text into French:\n\n{original_text}"}
]

print("Generating French translation...")
french_output = translator(messages_fr, max_new_tokens=100)

messages_de = [
    {"role": "system", "content": "You are a professional translator."},
    {"role": "user", "content": f"Translate this text into German:\n\n{original_text}"}
]

print("Generating German translation...")
german_output = translator(messages_de, max_new_tokens=100)


print("\n--- Original English Text ---")
print(original_text)

print("\n--- French Translation ---")
print(french_output[0]['generated_text'][-1]['content'].strip())

print("\n--- German Translation ---")
print(german_output[0]['generated_text'][-1]['content'].strip())

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating French translation...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating German translation...

--- Original English Text ---
Artificial intelligence is transforming the way we interact with technology. From natural language processing to computer vision, these advancements are making our daily tasks easier and more efficient.

--- French Translation ---
L'intelligence artificielle est transformant notre façon de nous intégrer avec le technologie. Des tâches langues naturelles à la vision numérique, ces avancées rendent nos tâches quotidiennes plus simples et plus efficaces.

--- German Translation ---
Die Artikulierung der Informatik wird in die Wirklichkeit verwandelt. Von natürlichen Sprachverarbeitung bis hin zu Computer Vision haben diese Anwendungen unsere tägliche Aufgaben leichter und effizienter gemacht.


**8. Implement a simple attention-based encoder-decoder model for
English-to-Spanish translation using Tensorflow or PyTorch.**

In [25]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# 1. Toy Dataset (English -> Spanish)
data_pairs = [
    ("the sky is blue", "el cielo es azul"),
    ("the sun is bright", "el sol es brillante"),
    ("the grass is green", "la hierba es verde"),
    ("the night is dark", "la noche es oscura")
]

# Helper function to build vocabulary
def build_vocab(sentences):
    vocab = {"<pad>": 0, "<sos>": 1, "<eos>": 2}
    for sentence in sentences:
        for word in sentence.split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

en_sentences, es_sentences = zip(*data_pairs)
en_vocab = build_vocab(en_sentences)
es_vocab = build_vocab(es_sentences)

en_idx2word = {i: w for w, i in en_vocab.items()}
es_idx2word = {i: w for w, i in es_vocab.items()}

# Vectorize and pad sequences
def preprocess(sentences, vocab, max_len=6):
    padded_data = []
    for s in sentences:
        tokens = [vocab["<sos>"]] + [vocab[w] for w in s.split()] + [vocab["<eos>"]]
        tokens += [vocab["<pad>"]] * (max_len - len(tokens))
        padded_data.append(tokens[:max_len])
    return torch.tensor(padded_data)

input_tensor = preprocess(en_sentences, en_vocab)
target_tensor = preprocess(es_sentences, es_vocab)

In [26]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=0)
        self.gru = nn.GRU(emb_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, hidden = self.gru(embedded)
        return outputs, hidden  # outputs: (batch, seq_len, hidden_dim)

class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W1 = nn.Linear(hidden_dim, hidden_dim)
        self.W2 = nn.Linear(hidden_dim, hidden_dim)
        self.V = nn.Linear(hidden_dim, 1)

    def forward(self, query, values):
        query = query.transpose(0, 1)  # (batch, 1, hidden_dim)

        # Calculate attention scores
        score = self.V(torch.tanh(self.W1(query) + self.W2(values)))  # (batch, seq_len, 1)
        attention_weights = F.softmax(score, dim=1)  # (batch, seq_len, 1)

        # Compute context vector
        context_vector = attention_weights * values  # (batch, seq_len, hidden_dim)
        context_vector = torch.sum(context_vector, dim=1, keepdim=True)  # (batch, 1, hidden_dim)

        return context_vector, attention_weights

class AttnDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim):
        super().__init__()
        self.attention = BahdanauAttention(hidden_dim)
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=0)
        self.gru = nn.GRU(emb_dim + hidden_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, input_token, hidden, encoder_outputs):
        # input_token: (batch, 1)
        embedded = self.embedding(input_token)  # (batch, 1, emb_dim)

        # Calculate context vector using attention
        context_vector, weights = self.attention(hidden, encoder_outputs)

        # Combine embedded input token and context vector
        rnn_input = torch.cat((embedded, context_vector), dim=2)  # (batch, 1, emb_dim + hidden_dim)

        output, hidden = self.gru(rnn_input, hidden)
        prediction = self.fc_out(output.squeeze(1))  # (batch, output_dim)

        return prediction, hidden, weights

In [27]:
# Model Hyperparameters
EMB_DIM = 16
HIDDEN_DIM = 32

encoder = Encoder(len(en_vocab), EMB_DIM, HIDDEN_DIM)
decoder = AttnDecoder(len(es_vocab), EMB_DIM, HIDDEN_DIM)

optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=0)

# Training execution
print("Training the Attention Model...")
for epoch in range(1, 151):
    optimizer.zero_grad()
    batch_size = input_tensor.size(0)
    max_target_len = target_tensor.size(1)

    encoder_outputs, encoder_hidden = encoder(input_tensor)
    decoder_hidden = encoder_hidden

    # Start sequence with <sos> token
    decoder_input = target_tensor[:, 0].unsqueeze(1)
    loss = 0

    for t in range(1, max_target_len):
        prediction, decoder_hidden, _ = decoder(decoder_input, decoder_hidden, encoder_outputs)
        loss += criterion(prediction, target_tensor[:, t])
        # Teacher Forcing: Feed the actual target token as the next input
        decoder_input = target_tensor[:, t].unsqueeze(1)

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}/150 | Total Sequence Loss: {loss.item():.4f}")

# Evaluation / Translation Step
print("\n--- Translating Verification Prompt ---")
encoder.eval()
decoder.eval()

with torch.no_grad():
    # Evaluate on the first sample: "the sky is blue"
    test_input = input_tensor[0:1]
    enc_outputs, enc_hidden = encoder(test_input)
    dec_hidden = enc_hidden

    dec_input = torch.tensor([[es_vocab["<sos>"]]])
    translated_words = []

    for _ in range(6):
        pred, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_outputs)
        next_token = pred.argmax(dim=1).item()

        if next_token == es_vocab["<eos>"] or next_token == es_vocab["<pad>"]:
            break
        translated_words.append(es_idx2word[next_token])
        dec_input = torch.tensor([[next_token]])

    print(f"English Input:     {data_pairs[0][0]}")
    print(f"Target Spanish:    {data_pairs[0][1]}")
    print(f"Model Translation: {' '.join(translated_words)}")

Training the Attention Model...
Epoch 50/150 | Total Sequence Loss: 0.0448
Epoch 100/150 | Total Sequence Loss: 0.0148
Epoch 150/150 | Total Sequence Loss: 0.0091

--- Translating Verification Prompt ---
English Input:     the sky is blue
Target Spanish:    el cielo es azul
Model Translation: el cielo es azul


**9. Use the following short poetry dataset to simulate poem generation with a
pre-trained GPT model:**

["Roses are red, violets are blue,",
"Sugar is sweet, and so are you.",
"The moon glows bright in silent skies,",
"A bird sings where the soft wind sighs."]

Using this dataset as a reference for poetic structure and language, generate a new 2-4 line poem using a pre-trained GPT model (such as GPT-2). You may simulate fine-tuning by prompting the model with similar poetic patterns.

In [31]:
from transformers import pipeline, set_seed

# Initialize the text-generation pipeline with GPT-2
generator = pipeline('text-generation', model='gpt2')

# Changed the seed to get a more poetic generation
set_seed(42)

prompt = (
    "Poem 1:\n"
    "Roses are red, violets are blue,\n"
    "Sugar is sweet, and so are you.\n\n"
    "Poem 2:\n"
    "The moon glows bright in silent skies,\n"
    "A bird sings where the soft wind sighs.\n\n"
    "Poem 3:\n"
)

print("Generating new poem...\n")

output = generator(
    prompt,
    max_new_tokens=50,     # Increased to ensure it finishes both lines
    temperature=0.7,       # Slightly lower temperature for better coherence
    num_return_sequences=1,
    pad_token_id=50256
)

full_text = output[0]['generated_text']
new_poem_raw = full_text.split("Poem 3:\n")[-1]

# Clean up the output by grabbing the first two non-empty lines
lines = [line.strip() for line in new_poem_raw.split('\n') if line.strip()]
new_poem = "\n".join(lines[:2])

print("--- Provided Dataset (Context) ---")
print("Roses are red, violets are blue,")
print("Sugar is sweet, and so are you.")
print("The moon glows bright in silent skies,")
print("A bird sings where the soft wind sighs.")
print("\n--- Generated Poem ---")
print(new_poem)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating new poem...

--- Provided Dataset (Context) ---
Roses are red, violets are blue,
Sugar is sweet, and so are you.
The moon glows bright in silent skies,
A bird sings where the soft wind sighs.

--- Generated Poem ---
A man lies in the woods,
He walks in a grassy plain


**10. Imagine you are building a creative writing assistant for a publishing
company. The assistant should generate story plots and character descriptions using
Generative AI. Describe how you would design the system, including model selection,
training data, bias mitigation, and evaluation methods. Explain the real-world challenges
you might face.**

1. Model Selection and Architecture
To protect the publishing company's proprietary manuscripts and intellectual property, relying solely on public APIs (like OpenAI) poses data privacy risks.

Base Model: An open-weights Large Language Model (LLM) with a strong foundation in instruction-following and long-context processing, such as Llama 3 (8B or 70B) or Mistral.

Deployment: The model should be containerized using Docker and deployed on scalable cloud infrastructure like AWS EC2. This ensures the publisher retains complete ownership of the data pipeline and model weights.

Architecture: Implement a Retrieval-Augmented Generation (RAG) framework. When a user requests a character or plot, the system retrieves relevant tropes, pacing structures, or successful historical examples from a vector database (e.g., MongoDB with vector search) before generation.

2. Training and Reference Data
A base model is too generic for professional publishing. It requires specialized data conditioning.

Pre-training / Base Data: The model inherently possesses knowledge of grammar, syntax, and public-domain literature (e.g., Shakespeare, Dickens).

Fine-Tuning Dataset: The company’s historical backlist of successful novels, categorized by genre, pacing, and character archetypes.

Metadata Tags: Structured data outlining character motivations (e.g., "The Reluctant Hero"), story beats (e.g., "Save the Cat" beats), and world-building constraints.

3. Bias Mitigation
Unchecked generative models often default to stereotypes, which can damage a publisher's reputation.

Curated Training Data: Actively audit the fine-tuning dataset to ensure representation across gender, ethnicity, and cultural backgrounds, removing historically prejudiced texts.

System Prompting: Inject hard constraints into the generation prompt (e.g., "Generate a character description that avoids cultural stereotypes and focuses on psychological depth").

Toxicity Classifiers: Use a secondary, lightweight NLP classifier layer to scan outputs for offensive, biased, or exclusionary language before presenting it to the writer.

4. Evaluation Methods
Evaluating creative text is subjective, making traditional metrics like BLEU or ROUGE nearly useless.

Human-in-the-Loop (HITL): The primary metric is acceptance rate—how often editors and authors actually use, accept, or lightly modify the generated plots and characters versus regenerating them.

Diversity Metrics: Algorithmic checks to measure the distribution of demographic markers in generated characters over time to ensure the model isn't defaulting to homogenous traits.

Coherence and Consistency Checking: Using an automated evaluator (a separate LLM instance) to cross-check if the generated plot contains logical plot holes or if the character description contradicts itself.

5. Real-World Challenges
Copyright and Plagiarism: The most severe risk is the model inadvertently memorizing and regurgitating copyrighted material from its training data. If an author publishes a generated plot that closely mirrors an existing intellectual property, the publisher faces massive legal liability.

Homogenization of Voice: If all authors at the company use the same AI tool to brainstorm, the publisher's books risk sounding identical, losing the unique authorial voices that drive sales.

Context Window Forgetting: Maintaining plot coherence over a complex, multi-threaded narrative requires massive memory. Models frequently "forget" a character's earlier traits or minor plot points as the context window fills up, leading to continuity errors.